# K513 · Week 5, Session 2
## Classification — decision trees

On Tuesday you handed the plant manager a list of twenty machines to pull off the line. This
morning she asks the obvious next question: **why those twenty? What is the rule?**

Tuesday's model cannot answer her in a sentence. Today's model can — and it turns out to be the
more accurate of the two, which is not the same as being the more useful.

Same 689 machines, same split, same crew of twenty.

---

### Before you type anything

**File → Save a copy in Drive.**

This notebook is read-only for you. You can type into it and run it and it will look completely
normal, but nothing you do will be saved. Save your own copy first, every time.

---

### Using AI in this notebook

Gemini is built into Colab and you are welcome to use it here. Two things worth knowing:

- It does not know which columns you have or what we covered in class. Whatever it writes, you own.
- The most useful thing you can ask it is **"explain what this line does"** — not "write it for me".

Today's trap: ask an AI for "the best `max_depth`" and it will loop over a range of depths, pick the
one with the highest **test** score, and report it as the answer. That is the one thing this course
has told you never to do. The AI cannot know that, because nothing in your code says so — the rule
lives in your head, not in the data.

---

### Turn off Unwanted AI Assistance

AI-powered coding completion is turned on by default. It is convenient but does not give you a chance
to think and learn. Turning it off helps you learn. You can always turn it back on when needed.
- Tools → settings → AI Assistance → Uncheck "Show AI-powered inline code completions"
- Tools → settings → Uncheck "Show context-powered code completions"

---

### How to run a cell

Click on a cell, then press **Shift + Enter**. That runs it and moves you to the next one.

## Setup

Everything below is Tuesday's notebook, plus one new import: `DecisionTreeClassifier`, and two
helpers for drawing and printing a tree.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text

pd.set_option('display.precision', 3)
np.set_printoptions(precision=3, suppress=True)

RANDOM_SEED = 42
MACHINE_URL = 'https://raw.githubusercontent.com/jl-uscn/k513-data/main/Machine%20Failure%20Data.csv'

### The data, and the same split as Tuesday

Nothing here is new. `random_state` and `stratify` are the same, so these are the same 516 training
and 173 test machines you worked with on Tuesday — which is the only reason the two models can be
compared at all.

In [ ]:
machine_df = pd.read_csv(MACHINE_URL)

X = machine_df.drop(columns=['Failure?', 'Product ID'])
y = (machine_df['Failure?'] == 'Yes').astype(int)

categorical_features = ['Type']
continuous_features = ['Air Temp', 'Proc Temp', 'Rtn Speed', 'Torque', 'Tool Wear']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_SEED, stratify=y)

print(f"training machines: {len(X_train)}   test machines: {len(X_test)}")
print(f"failures in the test set: {y_test.sum()}")

### Tuesday's model, rebuilt

Given complete, so you have something to compare against. This is exactly the model you built on
Tuesday — nothing about it has changed.

In [ ]:
logistic_pre = ColumnTransformer([
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_features),
    ('num', StandardScaler(), continuous_features)])

logistic_model = Pipeline([('preprocessor', logistic_pre),
                           ('classifier', LogisticRegression(random_state=RANDOM_SEED))])
logistic_model.fit(X_train, y_train)

print(f"Tuesday — train {logistic_model.score(X_train, y_train):.3f}"
      f"   test {logistic_model.score(X_test, y_test):.3f}")

### The twelve machines from the slide

The table you drew a line through in class, so you can check the counts for yourself.

In [ ]:
twelve = machine_df.loc[
    machine_df['Product ID'].isin(['L51882', 'M21683', 'M19525', 'L52100',
                                   'L54347', 'M22424', 'L56403', 'L51212',
                                   'L53296', 'M19720', 'M17019', 'L52006']),
    ['Product ID', 'Rtn Speed', 'Failure?']].sort_values('Rtn Speed')

twelve['side of 1381.5'] = np.where(twelve['Rtn Speed'] <= 1381.5, 'below', 'above')
pd.crosstab(twelve['side of 1381.5'], twelve['Failure?'])

Five of the six below the line failed; five of the six above it did not. **Two wrong out of
twelve**, and no other cut on this column does better.

---
## Section 1 — build the tree

▶ [Video walkthrough — what `ColumnTransformer` is doing, and why `'passthrough'` replaces the
scaler](https://youtu.be/VrfJT3pas9A)

The preprocessor below is Tuesday's, with one change: the numeric columns are **passed through**
instead of scaled. A tree only ever compares one column to a number inside that column, so it has no
opinion about units.

### ✏️ Now You Try · 1

**(a)** Fill in the two blanks: the preprocessing step for the numeric columns, and the classifier.

In [ ]:
tree_pre = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
    ('num', ____, continuous_features)])

tree_model = Pipeline([('preprocessor', tree_pre),
                       ('classifier', ____(max_depth=4, random_state=RANDOM_SEED))])
tree_model.fit(X_train, y_train)

**(b)** Print the training accuracy and the test accuracy.

In [ ]:
print(f"train {tree_model.score(____, ____):.3f}   test {tree_model.score(____, ____):.3f}")

**(c)** Put all three models side by side. The baseline is *always guess the class that comes
up more often in the training data* — work out what share of the **test** machines that would get
right.

In [ ]:
baseline = max(y_train.mean(), 1 - y_train.mean())
baseline_test = ____

pd.DataFrame([
    {'model': 'baseline',  'train': round(baseline, 3),
     'test': round(baseline_test, 3)},
    {'model': 'Tuesday — logistic',
     'train': round(logistic_model.score(X_train, y_train), 3),
     'test': round(logistic_model.score(X_test, y_test), 3)},
    {'model': 'tree, max_depth=4',
     'train': round(tree_model.score(X_train, y_train), 3),
     'test': round(tree_model.score(X_test, y_test), 3)},
]).set_index('model')

**(d)** Run **Steps to Evaluate a Model** on your two scores. Which step do you land on?

> ✏️ Answer in the cell below — this one is for the room, not for Canvas.

**Steps to Evaluate a Model** (Thursday of last week, slide 11):

1. Score on train and on test. Always both, always in that order.
2. Both low? Too simple. Give it more to work with.
3. Train high, test far below? Too much model for the data you have. Get more data, or add a
   penalty.
4. Close together, and better than the baseline? Stop. This is what this data has.
5. Never choose a setting because it scored best on the test set.

In [ ]:
# (d) which step, and what would you change first?

**(e)** In your own words: the tree's test score is **higher** than Tuesday's model *and* its
gap is **wider**. Are those two facts in conflict? Write one or two sentences.

In [ ]:
# (e) your sentence here

---
## Section 2 — how big should it be?

The helper below fits one tree per depth and reports both scores plus the number of leaves. **You
are not expected to be able to write it**, and it is not on any assessment. Read what it prints.

In [ ]:
def sweep_depths(depths):
    """Fit one tree per depth. Report both scores and how many rules it ended up with."""
    rows = []
    for d in depths:
        m = Pipeline([('preprocessor', tree_pre),
                      ('classifier', DecisionTreeClassifier(max_depth=d,
                                                            random_state=RANDOM_SEED))])
        m.fit(X_train, y_train)
        rows.append({'max_depth': d if d else 'none',
                     'leaves': m.named_steps['classifier'].get_n_leaves(),
                     'train': round(m.score(X_train, y_train), 3),
                     'test': round(m.score(X_test, y_test), 3)})
    out = pd.DataFrame(rows)
    out['gap'] = (out['train'] - out['test']).round(3)
    return out.set_index('max_depth')

### ✏️ Now You Try · 2

**(a)** Run the sweep. Add the full tree — pass `None` as the last depth — so you can see what
happens when nothing stops it.

In [ ]:
sweep_depths([1, 2, 3, 4, 5, 6, 8, 10, ____])

**(b)** **Which depth would you report to the plant manager, and why?**

Read the table before you answer. One depth has the highest test score in it — say what is wrong
with choosing that one for that reason.

In [ ]:
# (b) your answer here

**(c)** Refit at the depth you chose, then draw it.

▶ [Video walkthrough — reading a plotted tree](https://youtu.be/lyosW0cZMw4)

In [ ]:
final_tree = Pipeline([('preprocessor', tree_pre),
                       ('classifier', DecisionTreeClassifier(max_depth=____,
                                                             random_state=RANDOM_SEED))])
final_tree.fit(X_train, y_train)

feature_names = (list(final_tree.named_steps['preprocessor']
                      .named_transformers_['cat']
                      .get_feature_names_out(categorical_features))
                 + continuous_features)

plt.figure(figsize=(16, 8))
plot_tree(final_tree.named_steps['classifier'], feature_names=feature_names,
          class_names=['No', 'Yes'], filled=True, rounded=True,
          impurity=False, label='root', precision=1, fontsize=10)
plt.show()

**(d)** Follow one machine down the tree.

`machine_47` is the machine from Tuesday. Every one of its readings sits inside the range the model
was trained on — ask a model about a machine colder than anything it has ever seen and it will
still hand you a number, and that number is worth nothing.

Start at the top box, answer its question with `machine_47`'s readings, go left if the answer is
true and right if it is false, and keep going until you reach a box with no question.

In [ ]:
machine_47 = pd.DataFrame([{
    'Type': 'L', 'Air Temp': 300.0, 'Proc Temp': 310.0,
    'Rtn Speed': 1450, 'Torque': 40.0, 'Tool Wear': 100}])

print(f"the model says: {final_tree.predict_proba(machine_47)[0, 1]:.3f}")
machine_47

Now write that leaf's rule as **one sentence you would say out loud in a shift meeting** —
the readings that get you there, what the model concludes, and how many machines it has seen like
that.

▶ [Video walkthrough — printing the rules with `export_text()`](https://youtu.be/kQL9rlgTksA)

In [ ]:
print(export_text(final_tree.named_steps['classifier'],
                  feature_names=feature_names, show_weights=True))

# your sentence here

### Which columns did it actually use?

▶ [Video walkthrough — what feature importance measures](https://youtu.be/mHZP9twUjOQ)

In [ ]:
importances = pd.Series(final_tree.named_steps['classifier'].feature_importances_,
                        index=feature_names).sort_values()

ax = importances.plot(kind='barh', color='#2E5EA8')
ax.bar_label(ax.containers[0], fmt='%.3f', padding=4)
ax.set_xlim(0, 0.60)
ax.set_xlabel('share of the mess this column cleaned up')
plt.title('Two columns did nothing at all')
plt.show()

`Type` scores **exactly zero** — the tree never split on it, at any depth up to six. On
Tuesday, dropping `Type` *raised* the logistic model's test accuracy from 0.786 to 0.798.

Two models, two methods, the same verdict, and neither was asked. That is not a reason to regret
encoding it: **you cannot know a column is useless until you put it in.**

---
## Section 3 — which model goes in the memo?

Both models are fitted on the same 516 machines and scored on the same 173. Now compare what they
*produce*, not how well they score.

### ✏️ Now You Try · 3

**(a)** How many **different** probabilities does each model produce across the 173 test machines?

In [ ]:
logistic_p = logistic_model.predict_proba(X_test)[:, 1]
tree_p = final_tree.predict_proba(X_test)[:, 1]

print(f"logistic: {len(np.unique(logistic_p.round(6)))} different scores")
print(f"tree:     {len(np.unique(____.round(6)))} different scores")

**(b)** Sort the test machines by risk under each model and take the top 20 and the top 30.
How many of them really failed?

The helper keeps each machine's score next to what actually happened, so the two cannot come apart
when you sort.

In [ ]:
def top_n(scores, n):
    """Take the n highest-risk machines and count how many really failed."""
    ranked = pd.DataFrame({'score': scores, 'really failed': y_test.values})
    ranked = ranked.sort_values('score', ascending=False)
    cut = ranked['score'].iloc[n - 1]
    return {'pulled': n,
            'really failed': int(ranked.head(n)['really failed'].sum()),
            'score at the cut': round(cut, 3),
            'machines tied there': int((ranked['score'] == cut).sum())}


pd.DataFrame([{'model': 'logistic', **top_n(logistic_p, 20)},
              {'model': 'logistic', **top_n(logistic_p, 30)},
              {'model': 'tree',     **top_n(tree_p, ____)},
              {'model': 'tree',     **top_n(tree_p, ____)}]).set_index('model')

Look at the last column. At twenty machines the tie does not bite. At thirty it does.

**(c)** **Which model goes in the memo to the plant manager?**

There is no single right answer. What matters is that your reason and your number belong to each
other — name one figure from your own table above and say what it means for the decision.

Write three or four sentences.

In [ ]:
# (c) your recommendation here

---
## Where this leaves you

Nothing typed in this notebook is collected. The exit ticket is on the bottom half of your card.

**This week's homework covers both Tuesday and today** — it is a separate notebook, and the written
answers go into the Canvas quiz *Week 5 Homework - Classification*.